# Selector Group Chat

## Setup

In [1]:
import os
import re
from typing import List, Sequence
from dotenv import load_dotenv
from autogen_agentchat.agents import AssistantAgent, UserProxyAgent
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.messages import BaseAgentEvent, BaseChatMessage
from autogen_agentchat.teams import SelectorGroupChat
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient
import warnings
warnings.filterwarnings("ignore")

load_dotenv()
api_key = os.getenv("AICREDITS_API_KEY")
base_url = os.getenv("BASE_URL")

if not api_key:
    raise RuntimeError("Please set the AICREDITS_API_KEY environment variable.")

model_client = OpenAIChatCompletionClient(
    model="gpt-4o-mini", api_key=api_key, base_url=base_url, temperature=0
)

# Agent names used by the prompts and the routing functions
PLANNER = "PlanningAgent"
SEARCH = "WebSearchAgent"
ANALYST = "DataAnalystAgent"
USER_PROXY = "UserProxyAgent"
USER = "user"

## Tools

### real Serper-based tool

In [2]:
from langchain_community.utilities import GoogleSerperAPIWrapper

if not os.getenv("SERPER_API_KEY"):
    print("Warning: SERPER_API_KEY not set - the real search tool will fail.")

search_tool_wrapper = GoogleSerperAPIWrapper(type="search", k=10)

def search_web(query: str) -> str:
    """Search the web for the given query and return the results."""
    try:
        return search_tool_wrapper.run(query)
    except Exception as e:
        return f"SEARCH ERROR: {e}. No results."

### Analyst tools

In [3]:
def percentage_change_tool(start: float, end: float) -> float:
    """Percentage change from start to end: (end - start) / start * 100, rounded to 2 decimals.
    start must be > 0 and end must be >= 0."""
    if start <= 0 or end < 0:
        raise ValueError(f"Invalid input: start={start}, end={end} (need start > 0 and end >= 0)")
    return round(((end - start) / start) * 100, 2)


def per_game_to_total_tool(per_game: float, games: int) -> int:
    """Estimate a season TOTAL from a per-game average and games played (per_game * games, rounded).
    Only use when the exact total is not available; the result is an estimate."""
    if per_game < 0 or games <= 0:
        raise ValueError(f"Invalid input: per_game={per_game}, games={games}")
    return round(per_game * games)

## Agents

In [4]:
PLANNER_PROMPT = """
You are a planning agent. You only plan, delegate and write the final answer. You never search
and you never calculate yourself, and you never use your own knowledge for facts or numbers.

Your team members are:
    WebSearchAgent: searches the web for facts (it cannot calculate)
    DataAnalystAgent: calculates only on numbers it is given (it cannot search)

When assigning tasks, use this format (one line per step):
1. <agent> : <task>

Rules:
- If a search is still NOT FOUND after two retry, do not assign it again. In the final answer say the requested calculation could not be done.
- After workers report: if something is "NOT FOUND" or "MISSING DATA", write a NEW short plan with only the missing steps and the agent who must do them.
- If something is still NOT FOUND after retries, say clearly in the final answer that it could not be verified. Do not guess.
- Do NOT write TERMINATE in a plan.
- Only after WebSearchAgent AND DataAnalystAgent have both reported usable results, write a short final answer that uses ONLY the numbers they reported (copy the analyst's percentage exactly; say if a total is an estimate) and end that message with the word TERMINATE.
"""

SEARCH_PROMPT = """
You are a web search agent. Use your search tool to find the information assigned to you in the plan.

Rules:
- Report a number only if the snippet clearly names the player, the season and the exact stat (rebounds, not assists). Otherwise write "NOT FOUND". Also report games played whenever the results show it.
- Report ONLY facts that literally appear in the search results. Always include the context: season, player, and whether a number is a season TOTAL or a PER-GAME average (and games played if shown).
- Never fill gaps from your own knowledge and never calculate anything.
- If the results do not contain a requested figure, write "NOT FOUND: <what>" and try a differently-worded query on your next turn.
- End EVERY message with exactly one last line:
  SEARCH_DONE      -> all lookups assigned to you are answered, or definitively NOT FOUND
  SEARCH_CONTINUE  -> more lookups or retries are still needed
"""

ANALYST_PROMPT = """
You are a data analyst agent. Your tools: percentage_change_tool and per_game_to_total_tool.

Rules:
- Every percentage must come from percentage_change_tool output. Games played must come from the search report. If either is missing, reply "MISSING DATA"
- Use ONLY numbers that appear in the conversation (reported by WebSearchAgent). Never use your own knowledge.
- Never compare different stats (e.g. rebounds vs assists) or different seasons than the ones asked.
- If a needed number is missing, do NOT guess and do NOT call a tool. Reply exactly: "MISSING DATA: <what is missing, for which season>".
- Otherwise report briefly: the inputs you used (with season), the tool result, and whether any input was an estimate.
"""


def build_agents(search_tool):
    planning_agent = AssistantAgent(
        name=PLANNER,
        description="An agent for planning tasks, this agent should be the first to engage when given a new task.",
        model_client=model_client,
        system_message=PLANNER_PROMPT,
    )
    web_search_agent = AssistantAgent(
        name=SEARCH,
        description="An agent for searching the web for information.",
        model_client=model_client,
        tools=[search_tool],
        reflect_on_tool_use=True,
        system_message=SEARCH_PROMPT,
    )
    data_analyst_agent = AssistantAgent(
        name=ANALYST,
        description="An agent for performing calculations and data analysis.",
        model_client=model_client,
        tools=[percentage_change_tool, per_game_to_total_tool],
        reflect_on_tool_use=True,
        system_message=ANALYST_PROMPT,
    )
    return planning_agent, web_search_agent, data_analyst_agent

## Termination condition

In [5]:
def make_termination(max_messages: int = 20):
    return TextMentionTermination("TERMINATE") | MaxMessageTermination(max_messages=max_messages)

## Selector prompt, team factory and task

In [6]:
selector_prompt = """
Select an agent to perform the next task.

{roles}

Current conversation history:
{history}

Read the conversation, then select ONE agent from {participants} to speak next.
Rules:
- PlanningAgent speaks first and assigns tasks.
- Choose the agent that the latest plan gives the next unfinished step to.
  WebSearchAgent must finish its lookups before DataAnalystAgent calculates.
- If WebSearchAgent's last message ends with SEARCH_CONTINUE, choose WebSearchAgent again.
- If DataAnalystAgent reported a result or MISSING DATA, or all work is finished, choose PlanningAgent.
Only select one agent.
"""

In [ ]:
def build_team(search_tool, extra_participants=(), max_messages=40, **team_kwargs):
    """Builds a NEW team"""
    planning_agent, web_search_agent, data_analyst_agent = build_agents(search_tool)
    return SelectorGroupChat(
        participants=[planning_agent, web_search_agent, data_analyst_agent, *extra_participants],
        model_client=model_client,
        termination_condition=make_termination(max_messages),
        selector_prompt=selector_prompt,
        allow_repeated_speaker=True,
        **team_kwargs,
    )


task = (
    "Who was the NBA player with the most points for the Miami Heat in the 2006-2007 season, "
    "and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?"
)

## real web search

In [8]:
team = build_team(search_web)
result = await Console(team.run_stream(task=task))

---------- TextMessage (user) ----------
Who was the NBA player with the most points for the Miami Heat in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?
---------- TextMessage (PlanningAgent) ----------
1. WebSearchAgent : Search for the NBA player with the most points for the Miami Heat in the 2006-2007 season.
2. DataAnalystAgent : Calculate the percentage change in total rebounds for the identified player between the 2007-2008 and 2008-2009 seasons.
---------- ToolCallRequestEvent (WebSearchAgent) ----------
[FunctionCall(id='call_Z3SCGOPf1RHOygNG1RgHusPP', arguments='{"query":"Miami Heat player with most points 2006-2007 season"}', name='search_web')]
---------- ToolCallExecutionEvent (WebSearchAgent) ----------
[FunctionExecutionResult(content="Record: 44-38, Finished 1st in NBA Southeast Division ; Coach: Pat Riley (44-38) ; Executive: Randy Pfund ; PTS/G: 94.6 (28th of 30) Opp PTS/G: 95.5 (8th of 30). 

In [9]:
print("\nstop_reason:", result.stop_reason, "| messages:", len(result.messages))


stop_reason: Text 'TERMINATE' mentioned | messages: 15


## Save / inspect team state
Quick leak check: each agent's memory should contain **only this run**.

In [10]:
state = await team.save_state()
state

{'type': 'TeamState',
 'version': '1.0.0',
 'agent_states': {'PlanningAgent': {'type': 'ChatAgentContainerState',
   'version': '1.0.0',
   'agent_state': {'type': 'AssistantAgentState',
    'version': '1.0.0',
    'llm_context': {'messages': [{'content': 'Who was the NBA player with the most points for the Miami Heat in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?',
       'source': 'user',
       'type': 'UserMessage'},
      {'content': '1. WebSearchAgent : Search for the NBA player with the most points for the Miami Heat in the 2006-2007 season.\n2. DataAnalystAgent : Calculate the percentage change in total rebounds for the identified player between the 2007-2008 and 2008-2009 seasons.',
       'thought': None,
       'source': 'PlanningAgent',
       'type': 'AssistantMessage'},
      {'content': 'NOT FOUND: player with the most points for the Miami Heat in the 2006-2007 season. \n\nSEARCH_CONTINUE',
 

In [11]:
await team.reset()

In [12]:
def chat_only(messages: Sequence[BaseAgentEvent | BaseChatMessage]) -> List[BaseChatMessage]:
    """Keep only real chat messages (drop tool-call events)."""
    return [m for m in messages if isinstance(m, BaseChatMessage)]

def next_agent(messages: Sequence[BaseAgentEvent | BaseChatMessage]) -> str | None:
    """Common routing logic. Returns the next agent's name, or None if unclear."""
    last = chat_only(messages)[-1]
    text = last.to_text()

    if last.source == USER:
        print("##########1")
        return PLANNER

    if last.source == PLANNER:
        if SEARCH in text:
            print("##########2")
            return SEARCH
        if ANALYST in text:
            print("##########2")
            return ANALYST
        return None

    if last.source == SEARCH:
        return SEARCH if "SEARCH_CONTINUE" in text.upper() else ANALYST

    if last.source == ANALYST:
        print("##########FINAL")
        return PLANNER

    return None

## custom selector function
Deterministic routing for the known steps

In [13]:
def my_selector_fun(messages):
    return next_agent(messages)


team = build_team(search_web, selector_func=my_selector_fun)
result = await Console(team.run_stream(task=task))

##########1
---------- TextMessage (user) ----------
Who was the NBA player with the most points for the Miami Heat in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?
---------- TextMessage (PlanningAgent) ----------##########2

1. WebSearchAgent : Search for the NBA player with the most points for the Miami Heat in the 2006-2007 season.
2. DataAnalystAgent : Calculate the percentage change in total rebounds for the identified player between the 2007-2008 and 2008-2009 seasons.
---------- ToolCallRequestEvent (WebSearchAgent) ----------
[FunctionCall(id='call_8IHVitXyiJlByAqwMUllIlRz', arguments='{"query":"Miami Heat player with most points 2006-2007 season"}', name='search_web')]
---------- ToolCallExecutionEvent (WebSearchAgent) ----------
[FunctionExecutionResult(content="Record: 44-38, Finished 1st in NBA Southeast Division ; Coach: Pat Riley (44-38) ; Executive: Randy Pfund ; PTS/G: 94.6 (28th of 30) Opp P

In [14]:
print("\nstop_reason:", result.stop_reason, "| messages:", len(result.messages))


stop_reason: Text 'TERMINATE' mentioned | messages: 34


## human-in-the-loop with UserProxyAgent
Type **APPROVE** when asked to approve the plan. Approval is asked **once** (not after every worker turn).

In [ ]:
user_proxy_agent = UserProxyAgent(
    USER_PROXY, description="A proxy for the user to approve or disapprove the plan."
)


def selector_func_with_user_proxy(messages: Sequence[BaseAgentEvent | BaseChatMessage]) -> str | None:
    chat = chat_only(messages)
    last = chat[-1]

    approved = any(
        m.source == USER_PROXY and m.to_text().strip().upper().startswith("APPROVE")
        for m in chat
    )

    if last.source == USER_PROXY:
        if not last.to_text().strip().upper().startswith("APPROVE"):
            return PLANNER                       
        return next_agent(messages[:-1])

    if last.source == PLANNER and not approved and "TERMINATE" not in last.to_text().upper():
        return USER_PROXY

    return next_agent(messages)


team = build_team(
    search_web,
    extra_participants=[user_proxy_agent],
    selector_func=selector_func_with_user_proxy,
)
result = await Console(team.run_stream(task=task))

##########1
---------- TextMessage (user) ----------
Who was the NBA player with the most points for the Miami Heat in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?
---------- TextMessage (PlanningAgent) ----------
1. WebSearchAgent : Search for the NBA player with the most points for the Miami Heat in the 2006-2007 season.
2. DataAnalystAgent : Calculate the percentage change in total rebounds for the identified player between the 2007-2008 and 2008-2009 seasons.
---------- TextMessage (UserProxyAgent) ----------
##########2
Approve
---------- ToolCallRequestEvent (WebSearchAgent) ----------
[FunctionCall(id='call_ZTMO3fiVA9Nt7NaoDYKy8gbP', arguments='{"query":"Miami Heat player most points 2006-2007 season"}', name='search_web')]
---------- ToolCallExecutionEvent (WebSearchAgent) ----------
[FunctionExecutionResult(content="Record: 44-38, Finished 1st in NBA Southeast Division Coach: Pat Riley (44-38) Execu

In [16]:
print("\nstop_reason:", result.stop_reason, "| messages:", len(result.messages))


stop_reason: Text 'TERMINATE' mentioned | messages: 17
